# 🚀 OpenMythos-BerkahKarya: Finance Model Training

**Train a smart trading AI on Google Colab (FREE T4 GPU)**

This notebook trains OpenMythos 1B on comprehensive finance data:
- Trading analysis (XAUUSD, forex, crypto)
- Business plan generation
- Ad copy optimization
- Cashflow management
- Indonesian market specifics

**Runtime:** ~30-60 minutes on T4  
**VRAM:** ~8GB (QLoRA mode)  
**Output:** Finance-specialized LoRA adapter

In [ ]:
#@title 1. Setup Environment { display-mode: "form" }
!pip install torch transformers datasets accelerate -q
!git clone https://github.com/oyi77/OpenMythos-BerkahKarya.git
%cd OpenMythos-BerkahKarya
!pip install -e . -q

import torch
print(f"\n✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"✅ PyTorch: {torch.__version__}")

In [ ]:
#@title 2. Load Model { display-mode: "form" }
from open_mythos import OpenMythos, mythos_1b
from open_mythos.quantization import quantize_model

print("Loading mythos_1b...")
cfg = mythos_1b()
model = OpenMythos(cfg)

# QLoRA: INT4 quantization for 8GB VRAM
print("Applying INT4 quantization (QLoRA mode)...")
model = quantize_model(model, bits=4, group_size=128)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded: {total_params:,} parameters")

In [ ]:
#@title 3. Apply LoRA { display-mode: "form" }
from open_mythos.lora import LoRAConfig, apply_lora, print_lora_summary

# LoRA config optimized for finance
lora_config = LoRAConfig(
    rank=32,           # Higher rank = more capacity
    alpha=64,          # Scaling factor
    dropout=0.05,      # Regularization
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj"],
)

model = apply_lora(model, lora_config)
print_lora_summary(model)

model = model.cuda()
print(f"\n✅ Model ready on GPU")

In [ ]:
#@title 4. Generate Finance Training Data { display-mode: "form" }
# This cell uses the pre-generated dataset from the repo
import json

# Generate fresh data
!python3 data/generate_finance_data.py

# Load training data
train_data = []
with open("data/finance/train.jsonl") as f:
    for line in f:
        train_data.append(json.loads(line))

val_data = []
with open("data/finance/val.jsonl") as f:
    for line in f:
        val_data.append(json.loads(line))

print(f"✅ Training samples: {len(train_data)}")
print(f"✅ Validation samples: {len(val_data)}")
print(f"\nSample:\n{train_data[0]['text'][:200]}...")

In [ ]:
#@title 5. Prepare Dataset { display-mode: "form" }
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

class FinanceDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=1024):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        text = self.data[idx]['text']
        enc = self.tokenizer(
            text,
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        ids = enc['input_ids'].squeeze()
        mask = enc['attention_mask'].squeeze()
        return {'input_ids': ids, 'labels': ids.clone(), 'attention_mask': mask}

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

train_dataset = FinanceDataset(train_data, tokenizer, max_length=1024)
val_dataset = FinanceDataset(val_data, tokenizer, max_length=1024)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=0)

print(f"✅ DataLoader ready: {len(train_loader)} batches")

In [ ]:
#@title 6. Train! { display-mode: "form" }
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

# Only train LoRA params
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable, lr=2e-4, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=len(train_loader) * 5, eta_min=2e-5)

EPOCHS = 5
best_loss = 999

print(f"🚀 Starting training for {EPOCHS} epochs...")
print(f"📊 Trainable: {sum(p.numel() for p in trainable):,} parameters")
print(f"📊 Batches per epoch: {len(train_loader)}")
print()

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    start = time.time()
    
    for i, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].cuda()
        labels = batch['labels'].cuda()
        
        output = model(input_ids, labels=labels)
        loss = output.loss
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()
        scheduler.step()
        
        epoch_loss += loss.item()
        
        if (i + 1) % 20 == 0:
            avg = epoch_loss / (i + 1)
            lr = scheduler.get_last_lr()[0]
            print(f"  Epoch {epoch+1}/{EPOCHS} | Step {i+1}/{len(train_loader)} | Loss: {loss.item():.4f} | Avg: {avg:.4f} | LR: {lr:.6f}")
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].cuda()
            labels = batch['labels'].cuda()
            output = model(input_ids, labels=labels)
            val_loss += output.loss.item()
    val_loss /= len(val_loader)
    
    elapsed = time.time() - start
    avg_loss = epoch_loss / len(train_loader)
    print(f"\n📈 Epoch {epoch+1}/{EPOCHS} | Train: {avg_loss:.4f} | Val: {val_loss:.4f} | Time: {elapsed:.1f}s")
    
    if val_loss < best_loss:
        best_loss = val_loss
        from open_mythos.lora import save_lora_adapter
        save_lora_adapter(model, 'best_finance_adapter.pt', config=lora_config)
        print(f"  💾 Saved best adapter (val_loss: {val_loss:.4f})")
    print()

print("\n✅ Training complete!")
print(f"📊 Best validation loss: {best_loss:.4f}")

In [ ]:
#@title 7. Save Final Adapter { display-mode: "form" }
from open_mythos.lora import save_lora_adapter
import os

save_lora_adapter(model, 'openmythos-finance-v1.pt', config=lora_config)

size_mb = os.path.getsize('openmythos-finance-v1.pt') / 1024 / 1024
print(f"\n✅ Adapter saved: openmythos-finance-v1.pt ({size_mb:.1f} MB)")
print(f"\n📤 Upload to HuggingFace:")
print(f"   huggingface-cli upload oyi77/OpenMythos-Finance openmythos-finance-v1.pt")
print(f"\n📥 Download from Colab:")
print(f"   Files panel (left) → openmythos-finance-v1.pt → Download")

In [ ]:
#@title 8. Test the Model! { display-mode: "form" }
model.eval()

test_prompts = [
    "Analyze XAUUSD for trading opportunity. Current price: $2,350.",
    "Create a business plan for an e-commerce platform targeting Indonesian SMEs.",
    "Write ad copy for Meta Ads campaign promoting a trading course.",
    "Analyze cashflow: Revenue IDR 500M, Expenses IDR 400M, Balance IDR 1B.",
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*60}")
    
    enc = tokenizer(prompt, return_tensors='pt').input_ids.cuda()
    with torch.no_grad():
        output = model.generate(enc, max_new_tokens=200, n_loops=4)
    
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"\nOUTPUT:\n{generated}")

print(f"\n{'='*60}")
print("✅ Finance model is working!")

## 🎉 Done!

You now have a finance-specialized OpenMythos model!

### What's next:
1. **Upload adapter to HuggingFace** for community sharing
2. **Try different LoRA ranks** (8, 16, 32, 64) for quality vs speed
3. **Add your own training data** for specific use cases
4. **Export to GGUF** for local inference (ollama, llama.cpp)

### Links:
- [OpenMythos-BerkahKarya](https://github.com/oyi77/OpenMythos-BerkahKarya)
- [Upstream OpenMythos](https://github.com/kyegomez/OpenMythos)

### Support:
- GitHub Issues for bugs/features
- PRs welcome!